In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, FloatSlider, RadioButtons, HBox, Layout, VBox, HTML, GridBox
from IPython.display import display

# ============================================================
# RANDOM PROCESS PARAMETERS
# ============================================================

np.random.seed(32)

N_max = 1000
w = np.random.randn(N_max)
phi = np.random.uniform(0, 2*np.pi)

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_autocorrelation(process_type='White noise', N=500, max_lag=80, rho=0.85):

    if process_type == 'White noise':
        x = w[:N].copy()
        process_name = 'White Noise'

    elif process_type == 'Correlated AR(1)':
        x = np.zeros(N)
        x[0] = w[0]

        for i in range(1, N):
            x[i] = rho * x[i-1] + w[i]

        x = x / np.std(x)
        process_name = f'Correlated AR(1) Process, ρ = {rho:.2f}'

    else:
        n = np.arange(N)
        omega0 = 0.08 * np.pi
        x = np.cos(omega0 * n + phi)
        process_name = 'Random-Phase Sinusoidal Process'

    x_centered = x - np.mean(x)

    full_corr = np.correlate(x_centered, x_centered, mode='full')
    center = len(full_corr) // 2
    R = full_corr[center:center + max_lag + 1]

    normalization = np.arange(N, N - max_lag - 1, -1)
    R = R / normalization
    R = R / R[0]

    lags = np.arange(max_lag + 1)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.0, 6.2))

    ax1.plot(np.arange(N), x, linewidth=1.1)
    ax1.set_xlim(0, N - 1)

    if process_type == 'Random sinusoid':
        ax1.set_ylim(-1.4, 1.4)
    else:
        ax1.set_ylim(-4.0, 4.0)

    ax1.set_xlabel('Time index n', fontsize=12)
    ax1.set_ylabel('x[n]', fontsize=12)
    ax1.set_title(process_name, fontsize=13, pad=10)
    ax1.tick_params(axis='both', labelsize=10)
    ax1.grid(True, linestyle=':', alpha=0.6)

    ax2.plot(lags, R, linewidth=2)
    ax2.axhline(0, color='k', linewidth=1)
    ax2.axvline(0, color='k', linestyle='--', linewidth=1)

    ax2.set_xlim(0, max_lag)
    ax2.set_ylim(-1.1, 1.1)

    ax2.set_xlabel('Lag k', fontsize=12)
    ax2.set_ylabel('Normalized autocorrelation', fontsize=12)
    ax2.set_title('Autocorrelation Function', fontsize=13, pad=10)
    ax2.tick_params(axis='both', labelsize=10)
    ax2.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

# ============================================================
# RADIO BUTTONS
# ============================================================

process_selector = RadioButtons(
    options=['White noise', 'Correlated AR(1)', 'Random sinusoid'],
    value='White noise',
    description='Process:',
    style={'description_width': 'initial'},
    layout=Layout(width='270px')
)

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='90px')

N_slider = IntSlider(
    min=100,
    max=1000,
    step=50,
    value=500,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

lag_slider = IntSlider(
    min=20,
    max=200,
    step=10,
    value=80,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

rho_slider = FloatSlider(
    min=0.0,
    max=0.98,
    step=0.02,
    value=0.85,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

# ============================================================
# MAXIMUM VALUE LABELS
# ============================================================

N_max_label = HTML('<div style="font-family:Arial; font-size:14px;">1000</div>')

lag_max_label = HTML('<div style="font-family:Arial; font-size:14px;">200</div>')

rho_max_label = HTML('<div style="font-family:Arial; font-size:14px;">0.98</div>')

# ============================================================
# ENABLE / DISABLE rho SLIDER
# ============================================================

def update_rho_slider(change):
    rho_slider.disabled = (process_selector.value != 'Correlated AR(1)')

process_selector.observe(update_rho_slider, names='value')

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_autocorrelation,
    process_type=process_selector,
    N=N_slider,
    max_lag=lag_slider,
    rho=rho_slider
)

# ============================================================
# THEORY
# ============================================================

theory_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 16px;
    line-height: 1.5;
    width: 1050px;
">

<div style="
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 10px;
">
Autocorrelation as Memory of a Random Process
</div>

The autocorrelation function measures the statistical similarity
between a random process and a delayed version of itself.

<br><br>

<b>White noise:</b> essentially no memory for nonzero lags.

<br><br>

<b>Correlated AR(1):</b> neighboring samples are statistically related.
The parameter <b>ρ</b> determines the strength of this dependence.

<br><br>

<b>Random sinusoid:</b> periodic structure produces periodic autocorrelation.

<br><br>

For the AR(1) process:

<br><br>

ρ ≈ 0 &nbsp;&nbsp; → &nbsp;&nbsp; very short memory<br>

ρ large &nbsp;&nbsp; → &nbsp;&nbsp; slowly decaying autocorrelation and longer memory

<br><br>

The <b>AR parameter ρ</b> slider is active only when
<b>Correlated AR(1)</b> is selected.

</div>
""")

# ============================================================
# LEFT-ALIGNED LABELS
# ============================================================

N_label = HTML('<div style="font-family:Arial; font-size:14px;">Samples N:</div>')

lag_label = HTML('<div style="font-family:Arial; font-size:14px;">Maximum lag:</div>')

rho_label = HTML('<div style="font-family:Arial; font-size:14px;">AR parameter ρ:</div>')

# ============================================================
# FIXED THREE-COLUMN GRID
# LABEL | SLIDER | MAXIMUM VALUE
# ============================================================

slider_grid = GridBox(
    children=[
        N_label, N_slider, N_max_label,
        lag_label, lag_slider, lag_max_label,
        rho_label, rho_slider, rho_max_label
    ],
    layout=Layout(
        width='255px',
        grid_template_columns='110px 90px 43px',
        grid_template_rows='30px 30px 30px',
        grid_gap='2px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# EXTRA SPACE BETWEEN RADIO BUTTONS AND SLIDER GROUP
# ============================================================

slider_group = VBox(
    [slider_grid],
    layout=Layout(
        margin='12px 0px 0px 0px'
    )
)

# ============================================================
# CONTROLS
# ============================================================

controls = VBox(
    [
        process_selector,
        slider_group
    ],
    layout=Layout(
        width='270px',
        min_width='270px',
        align_items='flex-start',
        justify_content='center',
        margin='0px 0px 0px 10px',
        overflow='hidden'
    )
)

# ============================================================
# FIGURE LEFT - CONTROLS RIGHT
# ============================================================

graph_and_controls = HBox(
    [
        widget_plot.children[-1],
        controls
    ],
    layout=Layout(
        width='1100px',
        align_items='center',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_html,
        graph_and_controls
    ],
    layout=Layout(
        width='1100px',
        overflow='hidden'
    )
)

display(main_layout)